# PST-Parser su Colab

Questo notebook non contiene logica: clona il repository e invoca la stessa interfaccia a riga di
comando che si usa in locale e in container.

**Non serve alcuna credenziale.** Il repository è pubblico e il modello base è servito da un mirror
non soggetto ad accesso condizionato, quindi non vanno configurati secret.

Prima di eseguire:

1. **Runtime, Cambia tipo di runtime, GPU**. Una L4 basta e avanza: 24 GB di memoria.
2. Su Colab Pro, attivare anche **l'esecuzione in background**: il training dura un paio d'ore e
   senza quell'opzione il run muore quando si chiude la scheda.

Viene chiesta l'autorizzazione a Google Drive, usato solo per conservare la cache dei pesi del
modello fra un riavvio del runtime e l'altro, e per portare a casa gli artefatti alla fine.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
REPO = "https://github.com/gianfrancobarba/PST-Parser.git"
BRANCH = "main"

!git clone --branch {BRANCH} --depth 1 {REPO} /content/pst-parser
%cd /content/pst-parser

In [ ]:
# Colab ships its own build of torch, so no torch extra is requested here. The
# structured extra is what the schema-constrained run needs.
!pip install --quiet -e ".[train,unsloth,structured]"
!pstparser --version

In [ ]:
import os

from google.colab import drive

# The base model is served by a mirror that is not gated, so no Hugging Face
# credential is needed. The cache goes on Drive so the weights survive a runtime
# restart instead of being downloaded again.
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf-cache"

## Pipeline

I comandi girano con `configs/experiments/baseline.yaml` **così com'è**: su una GPU da 16 GB in su
la configurazione di riferimento entra in memoria, valutazione durante il training compresa.

Ogni comando accetta comunque `--set chiave.annidata=valore` per sovrascrivere un singolo
parametro. Su una GPU più piccola, dimezzare il batch e raddoppiare l'accumulo lascia invariata la
dimensione effettiva — e quindi la traiettoria di ottimizzazione — riducendo solo il picco di
memoria:

    --set training.per_device_train_batch_size=1 --set training.gradient_accumulation_steps=8

I valori sono interpretati come YAML: `no`, `yes`, `on` e `off` diventano booleani, quindi per
passarne uno come testo va virgolettato, ad esempio `--set 'training.eval_strategy="no"'`.


In [ ]:
CONFIG = "configs/experiments/baseline.yaml"

!pstparser prepare-data --config {CONFIG}

In [ ]:
!pstparser train --config {CONFIG}

In [ ]:
from pathlib import Path

# Training writes to a timestamped directory under outputs/. The timestamp
# prefix orders the names chronologically, so the last one is the run that just
# finished. Predictions land beside the adapter that produced them.
RUN_DIR = sorted(path for path in Path("outputs").iterdir() if path.is_dir())[-1]
print("run:", RUN_DIR)

!pstparser generate --config {CONFIG} --adapter {RUN_DIR}/adapter
!pstparser score --config {CONFIG} --predictions {RUN_DIR}/predictions.jsonl
!pstparser align --config {CONFIG} --predictions {RUN_DIR}/predictions.jsonl

## Le due esecuzioni di confronto

Un punteggio da solo dice quanto bene il compito è svolto, non quanto il
fine-tuning abbia aggiunto. Le due celle seguenti rispondono a quella domanda e
a quella sul formato, e costano generazione ma non training.

In [ ]:
# Il modello base, non addestrato, sullo stesso test set e con lo stesso system
# message. È il pavimento contro cui vanno letti i punteggi della run allineata.
!pstparser generate --config {CONFIG} --run-dir outputs/zero_shot
!pstparser score --config {CONFIG} --predictions outputs/zero_shot/predictions.jsonl --run-dir outputs/zero_shot
!pstparser align --config {CONFIG} --predictions outputs/zero_shot/predictions.jsonl --run-dir outputs/zero_shot

In [ ]:
# Decodifica vincolata allo schema: la validità JSON diventa garantita per
# costruzione invece che misurata, quindi in questa modalità smette di dire
# alcunché sul fine-tuning. Il valore sta nel confronto con la run libera.
!pstparser generate --config {CONFIG} --adapter {RUN_DIR}/adapter --run-dir outputs/constrained --set inference.structured_output=true
!pstparser score --config {CONFIG} --predictions outputs/constrained/predictions.jsonl --run-dir outputs/constrained
!pstparser align --config {CONFIG} --predictions outputs/constrained/predictions.jsonl --run-dir outputs/constrained

## Recupero degli artefatti

Il filesystem di Colab non sopravvive alla sessione. Si copia su Drive solo ciò
che serve conservare: l'adapter pesa pochi megabyte, e predizioni, metriche,
allineamenti e manifest bastano a ricalcolare e ad attribuire ogni numero in
seguito, anche senza GPU. I checkpoint intermedi pesano gigabyte e non
riproducono nulla che l'adapter non riproduca.

In [ ]:
import shutil
from pathlib import Path

KEEP = (
    "predictions.jsonl",
    "results.json",
    "eval_details.jsonl",
    "alignments.jsonl",
    "run_manifest.json",
    "training_metrics.jsonl",
)
DEST = Path("/content/drive/MyDrive/pst-parser-outputs")

shutil.copytree(RUN_DIR / "adapter", DEST / "adapter", dirs_exist_ok=True)
for path in Path("outputs").rglob("*"):
    if path.is_file() and path.name in KEEP:
        target = DEST / path.relative_to("outputs")
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)

print("copiato in", DEST)